In [1]:
# scripts/make_eda.py
"""
Excel → raw CSV → pending-aware clean CSV + long-form EDA
=========================================================

What this builds
----------------
1) A verbatim export of the Excel into CSV:
   - data/raw/data_raw.csv
2) A *pending-aware* clean CSV suitable for analysis and downstream visuals:
   - data/processed/data_clean.csv
   * For Y/N-like columns (MC, Core, WG1…):
       - We create a parallel “*_status” text column with values in {"Yes","No","Pending",<NA>,<Unknown>}.
       - We overwrite the original flag column with a **nullable boolean** (True/False/<NA>).
         • STRICT: only a single-token yes/no is accepted. Phrases like “yes but resigning”
           are kept verbatim in *_status and mapped to boolean <NA>.
       - We derive `any_wg` as **nullable boolean**:
           True  if any WG is True;
           <NA>  if none True but at least one WG is Pending **or Unknown**;
           False otherwise.
   * ITC countries are added from a maintained list (Yes/No) for a deterministic reference.
3) A long-form EDA report you can open in Excel/Sheets:
   - outputs/eda_overview.csv
     Sections include:
       - dtype changes (before vs after)
       - Y/N detection before cleaning
       - Y/N summary after cleaning (pending-aware)
       - per-column summary (dtype, missing, unique)
       - numeric/boolean describe()
       - top values for categorical/object columns
       - country counts

Why “pending-aware” + strict yes/no?
------------------------------------
Downstream charts need stable booleans, but we must not lose that some values are “Pending”
or ambiguous free-text. We:
  • keep the exact human-readable status in `*_status`, and
  • map status → a nullable boolean for analytics. Only exact single-token Yes/No are coerced.

Typical usage
-------------
python scripts/make_eda.py
python scripts/make_eda.py --input data/raw/participants.xlsx --sheet 0
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional
import argparse
import re
import sys

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Repo root detection (so this works from /scripts, repo root, or notebooks)
# ---------------------------------------------------------------------------
def _find_repo_root() -> Path:
    try:
        here = Path(__file__).resolve()
        return here.parent.parent  # …/scripts → repo root
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir():
            return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir():
            return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()):
                return cur
            cur = cur.parent
        return cwd


ROOT     = _find_repo_root()
RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUT_DIR  = ROOT / "outputs"
AUX_DIR  = ROOT / "data"
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root →", ROOT)


# ---------------------------------------------------------------------------
# Helpers: I/O, text cleaning, flag detection, status mapping, typing
# ---------------------------------------------------------------------------
def write_csv(df: pd.DataFrame, path: Path) -> Path:
    """Save a DataFrame to UTF-8 CSV (with BOM so Excel opens accents correctly)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved → {path.relative_to(ROOT)}")
    return path


def snake_case(name: str) -> str:
    """Normalise headers to lowercase_with_underscores (and strip odd chars)."""
    name = str(name).replace("\u00A0", " ")
    name = re.sub(r"[^\w\s\-]+", " ", name)
    name = " ".join(name.split()).strip().lower().replace("-", " ")
    return re.sub(r"\s+", "_", name)


def clean_text_series(s: pd.Series) -> pd.Series:
    """Gentle text clean for object columns; never ‘over-cleans’."""
    out = (
        s.astype(str)
         .str.replace(r"\*", "", regex=True)
         .str.replace("\u00A0", " ", regex=False)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )
    return out.replace({"": np.nan, "nan": np.nan})


# Columns that *likely* encode yes/no flags by name (expand as needed)
_YN_PATTERNS = [re.compile(p, re.I) for p in (
    r"^wg[\s_]*\d+",       # WG1, WG 1, WG_1 …
    r"^is_",               # is_active, is_member…
    r"\bmc\b|\bmc_member\b",
    r"\bcore\b|\bcore_group\b",
    r"^wg_member$",
)]


def looks_like_yn(names: Iterable[str]) -> list[bool]:
    """Heuristic: which columns *look* like Y/N flags by their names?"""
    return [any(p.search(str(c).lower()) for p in _YN_PATTERNS) for c in names]


# -------------------- STRICT status parser (Yes/No/Pending/Unknown) --------------------
def status_from_tokens(s: pd.Series) -> pd.Series:
    """
    Strictly map free-text to: "Yes" / "No" / "Pending" / <Unknown> / <NA>

    Rules:
      • Only exact single-token values are accepted as Yes/No: {"y","yes","true","1","member","x"} and {"n","no","false","0"}.
      • Phrases like "yes but resigning" are NOT Yes/No (kept verbatim → will map to boolean <NA>).
      • If text contains 'pending'/'tbc'/'awaiting' anywhere → "Pending".
      • Unknown strings are preserved (auditable) and will map to boolean <NA>.
    """
    raw = s.astype(str)
    stripped = raw.str.strip()
    low = stripped.str.lower()

    # Pending wins if it appears anywhere
    pend_mask = low.str.contains(r"\b(pending|tbc|awaiting)\b", na=False)

    # Reduce to a single token for strict matching (strip punctuation/extra spaces)
    token = (
        low.str.replace(r"[()\[\]{}.,;:!/?\-]+", " ", regex=True)
           .str.replace(r"\s+", " ", regex=True)
           .str.strip()
    )

    YES = {"y", "yes", "true", "1", "member", "x"}
    NO  = {"n", "no", "false", "0"}

    yes_mask = token.isin(YES)
    no_mask  = token.isin(NO)

    out = pd.Series(index=s.index, dtype="object")
    out[yes_mask & ~pend_mask] = "Yes"
    out[no_mask  & ~pend_mask] = "No"
    out[pend_mask]             = "Pending"

    # Keep unfamiliar tokens (auditable), or NA if blank
    others = out.isna()
    out[others] = stripped[others].where(stripped[others].ne(""), np.nan)

    # Pretty title-case for readability in the *_status columns
    return out.map(lambda v: v.title() if isinstance(v, str) else v)


def status_to_nullable_bool(s: pd.Series) -> pd.Series:
    """Map status → pandas nullable boolean: 'Yes'→True, 'No'→False, others/NA→<NA>."""
    return s.map({"Yes": True, "No": False}).astype("boolean")


def gentle_type_infer(s: pd.Series) -> pd.Series:
    """
    Conservative typing pass:
      1) If it *mostly* looks numeric → float/int (coerce bad rows to NA)
      2) If it *mostly* looks like dates → datetime64[ns] (coerce bad rows to NaT)
      3) Otherwise keep as is.
    """
    if s.dtype == "object":
        raw = s.astype(str).str.replace(",", "").str.strip()
        looks_num = raw.str.match(r"^-?\d+(\.\d+)?$", na=False)
        if looks_num.mean() >= 0.6:
            return pd.to_numeric(raw, errors="coerce")

    if s.dtype == "object" or s.dtype.kind in "Mm":
        if s.dtype.kind in "Mm":
            return s
        sample = s.astype(str).str.lower()
        looks_date = sample.str.contains(
            r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec",
            regex=True, na=False
        )
        if looks_date.mean() >= 0.5:
            parsed = pd.to_datetime(s, errors="coerce", dayfirst=True)
            if parsed.notna().mean() >= 0.5:
                return parsed
    return s


def clean_country_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 'country_clean' if there is a 'country' column:
      - Drops trailing parentheses (e.g., "Georgia (GE)"→"Georgia")
      - Applies light text clean
      - Normalises some common aliases
    """
    matches = [c for c in df.columns if c.lower() == "country"]
    if not matches:
        return df
    col = matches[0]
    df["country_clean"] = df[col].astype(str).str.replace(r"\(.*?\)", "", regex=True)
    df["country_clean"] = clean_text_series(df["country_clean"])

    aliases = {
        "uk": "United Kingdom", "united kingdom": "United Kingdom", "great britain": "United Kingdom",
        "czech republic": "Czechia",
        "turkey": "Türkiye", "turkiye": "Türkiye",
        "north macedonia": "North Macedonia", "macedonia": "North Macedonia",
        "bosnia and herzegovina": "Bosnia and Herzegovina",
        "moldova": "Moldova", "russia": "Russia",
        "ivory coast": "Côte d'Ivoire", "cote d ivoire": "Côte d'Ivoire",
        "republic of kosovo": "Kosovo", "kosovo": "Kosovo",
        "republic of serbia": "Serbia",
        "republic of north macedonia": "North Macedonia",
    }
    low = df["country_clean"].str.lower()
    mask = low.isin(aliases)
    df.loc[mask, "country_clean"] = low.map(aliases)
    return df


def value_counts_preview(s: pd.Series, n: int = 10) -> str:
    """Short, Excel-friendly preview for the EDA CSV."""
    vc = s.astype("string").fillna("<NA>").value_counts(dropna=False).head(n)
    return "; ".join(f"{k}: {int(v)}" for k, v in vc.items())


def mark_section(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Tag a DataFrame with a 'section' column so we can concat many into one CSV."""
    out = df.copy()
    out.insert(0, "section", name)
    return out


# ------------------- ITC support: load list + normalise names ----------------
def _norm_country(s: str) -> str:
    """Normalise for ITC list matching (lowercase + common simplifications)."""
    s = (str(s) or "").strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)
    s = s.replace("’", "'")
    s = " ".join(s.split()).lower()
    s = s.replace("republic of ", "")
    aliases_lc = {
        "czech republic": "czechia",
        "macedonia": "north macedonia",
        "turkey": "türkiye", "turkiye": "türkiye", "türkiye": "türkiye",
        "uk": "united kingdom",
        "cote d'ivoire": "côte d'ivoire",
    }
    return aliases_lc.get(s, s)


def load_itc_set(path: Path) -> set[str]:
    """Read data/itc_countries.txt → set of normalised country names."""
    raw = path.read_text(encoding="utf-8").splitlines()
    items = [_norm_country(x) for x in raw if str(x).strip()]
    return set(items)


# ---------------------------------------------------------------------------
# CLI / input selection
# ---------------------------------------------------------------------------
def find_default_excel(raw_dir: Path) -> Optional[Path]:
    """Prefer the known CA23107 file; else first .xlsx/.xls we find."""
    preferred = raw_dir / "CA23107participant list.xlsx"
    if preferred.exists():
        return preferred
    files = sorted(list(raw_dir.glob("*.xlsx")) + list(raw_dir.glob("*.xls")))
    return files[0] if files else None


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Build data_raw.csv, data_clean.csv, eda_overview.csv (pending-aware, strict yes/no)")
    p.add_argument("--input", "-i", type=str, default=None,
                   help="Excel file (default: CA23107… or first .xlsx/.xls in data/raw/)")
    p.add_argument("--sheet", "-s", default=0,
                   help="Sheet index (0) or name (str). Default: 0")
    p.add_argument("--topn", type=int, default=25,
                   help="Top-N values in EDA previews (default 25)")
    # Jupyter/IDE inject unknown args; be forgiving.
    if "ipykernel" in sys.modules or "IPython" in sys.modules:
        args, _ = p.parse_known_args()
    else:
        args = p.parse_args()
    return args


# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------
def main() -> None:
    args = parse_args()

    # Resolve input Excel path (repo-relative if not absolute)
    if args.input:
        excel_path = Path(args.input)
        if not excel_path.is_absolute():
            excel_path = (ROOT / excel_path).resolve()
    else:
        excel_path = find_default_excel(RAW_DIR) or Path()

    if not excel_path.exists():
        print("ERROR: Excel not found.")
        print(f"- Looked for: {RAW_DIR / 'CA23107participant list.xlsx'}")
        print(f"- Or first .xlsx/.xls in: {RAW_DIR}")
        print("Or pass: python scripts/make_eda.py --input data/raw/yourfile.xlsx")
        sys.exit(1)

    sheet = args.sheet
    top_n = args.topn

    # Friendly path print
    rel = excel_path
    try:
        rel = excel_path.relative_to(ROOT)
    except Exception:
        pass
    print(f"Reading Excel: {rel}")

    # 1) Read Excel as-is
    df_raw = pd.read_excel(excel_path, sheet_name=sheet)

    # 2) Save verbatim copy (for audits / diffs)
    write_csv(df_raw, RAW_DIR / "data_raw.csv")

    # 3) Detect Y/N-like columns BEFORE cleaning (based on original names)
    yn_flags_before = looks_like_yn(df_raw.columns)
    yn_candidates_raw = [c for c, is_yn in zip(df_raw.columns, yn_flags_before) if is_yn]
    yn_before_df = pd.DataFrame({
        "original_name": yn_candidates_raw,
        "normalized_name_if_any": [snake_case(c) for c in yn_candidates_raw],
        "dtype_raw": [str(df_raw[c].dtype) for c in yn_candidates_raw],
        "top_values_preview": [value_counts_preview(df_raw[c], n=10) for c in yn_candidates_raw],
    })

    # 4) Light cleaning
    df = df_raw.copy()

    # 4a) Normalise headers
    old_to_new = {c: snake_case(c) for c in df.columns}
    df.rename(columns=old_to_new, inplace=True)

    # 4b) Clean text columns
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        df[obj_cols] = df[obj_cols].apply(clean_text_series)

    # 4c) Country helper (adds 'country_clean' if 'country' exists)
    df = clean_country_column(df)

    # 4d) ITC countries (Yes/No) using data/itc_countries.txt if present
    itc_path = AUX_DIR / "itc_countries.txt"
    base_col = "country_clean" if "country_clean" in df.columns else ("country" if "country" in df.columns else None)
    if itc_path.exists() and base_col:
        itc_set = load_itc_set(itc_path)
        df["itc_countries"] = df[base_col].map(lambda x: "Yes" if _norm_country(x) in itc_set else "No")
        print(f"Loaded ITC list ({len(itc_set)} names) → added 'itc_countries' (Yes/No).")
    elif base_col:
        # Deterministic presence helps downstream code.
        df["itc_countries"] = "No"

    # 4e) PENDING-AWARE flags (STRICT)
    #     Find likely Y/N flag columns AFTER header cleaning
    yn_flags_after = looks_like_yn(df.columns)
    yn_cols_after = [c for c, is_yn in zip(df.columns, yn_flags_after) if is_yn]

    # 4e-i) Create *_status columns with {'Yes','No','Pending',<Unknown>,<NA>}
    status_cols = []
    for c in yn_cols_after:
        status_col = f"{c}_status"
        df[status_col] = status_from_tokens(df[c])
        status_cols.append(status_col)

    # Optional report: how many Unknown (non Yes/No/Pending) per flag
    unknown_report = {}
    for c in yn_cols_after:
        sc = f"{c}_status"
        st = df[sc].dropna()
        unknown_report[sc] = int((~st.isin(["Yes", "No", "Pending"])).sum())
    print("Unknown statuses (treated as <NA>):",
          {k: v for k, v in unknown_report.items() if v})

    # 4e-ii) Replace original flag columns with **nullable booleans** mapped from *_status
    for c in yn_cols_after:
        status_col = f"{c}_status"
        df[c] = status_to_nullable_bool(df[status_col])

    # 4e-iii) Derive any_wg as nullable boolean from all WG columns
    wg_cols = [c for c in yn_cols_after if re.match(r"(?i)^wg[\s_]*\d+", c)]
    if wg_cols:
        # True if *any* WG boolean is True (treat NAs as False for this check)
        any_true = df[wg_cols].apply(lambda r: bool(r.fillna(False).any()), axis=1)

        # Look at *_status columns to decide if we should mark <NA>
        wg_status_cols = [f"{c}_status" for c in wg_cols if f"{c}_status" in df.columns]
        if wg_status_cols:
            statuses = df[wg_status_cols]
            any_pending = statuses.eq("Pending").any(axis=1)
            # Unknown = any status not in {"Yes","No","Pending"} (excluding NaN)
            known = statuses.isin(["Yes", "No", "Pending"]) | statuses.isna()
            any_unknown = ~known.all(axis=1)
        else:
            any_pending = pd.Series(False, index=df.index)
            any_unknown = pd.Series(False, index=df.index)

        df["any_wg"] = pd.Series(
            np.where(
                any_true,
                True,
                np.where(any_pending | any_unknown, pd.NA, False)
            ),
            dtype="boolean"
        )
    elif "any_wg" not in df.columns:
        df["any_wg"] = pd.Series([False] * len(df), dtype="boolean")

    # 5) Conservative type inference for non-status columns
    #    (Leave *_status as text; keep nullable booleans as is)
    skip_cols = set(status_cols + yn_cols_after + ["any_wg", "itc_countries"])
    for c in df.columns:
        if c in skip_cols:
            continue
        df[c] = gentle_type_infer(df[c])

    # 6) Save cleaned CSV
    write_csv(df, PROC_DIR / "data_clean.csv")

    # 7) Build long-form EDA CSV (many sections stacked into one file)
    eda_parts: list[pd.DataFrame] = []

    # 7a) dtype changes (before vs after)
    dtype_changes = pd.DataFrame({
        "original_name": list(df_raw.columns),
        "cleaned_name": [old_to_new.get(c, c) for c in df_raw.columns],
        "dtype_raw": [str(df_raw[c].dtype) for c in df_raw.columns],
        "dtype_final": [str(df[old_to_new.get(c, c)].dtype) if old_to_new.get(c, c) in df.columns else "<missing>" for c in df_raw.columns],
    })
    dtype_changes["changed"] = dtype_changes["dtype_raw"] != dtype_changes["dtype_final"]
    eda_parts.append(mark_section(dtype_changes, "dtype_changes"))

    # 7b) Y/N detection before cleaning
    if not yn_before_df.empty:
        eda_parts.append(mark_section(yn_before_df, "yn_detection_before"))

    # 7c) Y/N summary after cleaning (pending-aware, strict)
    if yn_cols_after:
        rows = []
        for c in yn_cols_after:
            s  = df[c]  # nullable boolean
            sc = f"{c}_status"

            # Count logic that cleanly separates True / False / NA
            true_count  = int((s == True).sum())   # noqa: E712
            na_count    = int(s.isna().sum())
            non_na      = int(s.notna().sum())
            false_count = int(non_na - true_count)

            rows.append({
                "flag_col": c,
                "status_col": sc,
                "dtype_flag": str(s.dtype),
                "true_count": true_count,
                "false_count": false_count,
                "na_count": na_count,
                "status_preview": value_counts_preview(df[sc], n=10),
            })
        eda_parts.append(mark_section(pd.DataFrame(rows), "yn_after_pending_aware"))

    # 7d) columns summary
    cols_summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [int(df[c].notna().sum()) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    }).sort_values("column")
    eda_parts.append(mark_section(cols_summary, "columns_summary"))

    # 7e) numeric/boolean summary (long format)
    num_cols = df.select_dtypes(include=[np.number, "boolean"]).columns
    if len(num_cols):
        desc_long = (
            df[num_cols].describe(include="all")
                        .T.reset_index().rename(columns={"index": "column"})
                        .melt(id_vars="column", var_name="metric", value_name="value")
        )
        eda_parts.append(mark_section(desc_long, "numeric_summary"))

    # 7f) top values for object/category columns (includes *_status)
    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    if len(cat_cols):
        rows = []
        for c in cat_cols:
            vc = df[c].astype("string").fillna("<NA>").value_counts(dropna=False).head(top_n)
            rows.extend({"column": c, "value": k, "count": int(v)} for k, v in vc.items())
        eda_parts.append(mark_section(pd.DataFrame(rows), "top_values"))

    # 7g) country counts (if present)
    country_col = [c for c in df.columns if c == "country_clean"] or [c for c in df.columns if c == "country"]
    if country_col:
        country_counts = (
            df[country_col[0]].astype("string").fillna("<NA>")
              .value_counts(dropna=False).rename_axis("country")
              .reset_index(name="count")
        )
        eda_parts.append(mark_section(country_counts, "countries_counts"))

    # Write the stacked EDA CSV
    eda_overview = pd.concat(eda_parts, ignore_index=True, sort=False) if eda_parts else pd.DataFrame({"section":[]})
    write_csv(eda_overview, OUT_DIR / "eda_overview.csv")

    print("\nAll files written under:")
    print(" -", RAW_DIR.relative_to(ROOT))    # data_raw.csv
    print(" -", PROC_DIR.relative_to(ROOT))   # data_clean.csv (pending-aware, strict)
    print(" -", OUT_DIR.relative_to(ROOT))    # eda_overview.csv")


if __name__ == "__main__":
    main()


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood
Reading Excel: data\raw\CA23107participant list.xlsx
Saved → data\raw\data_raw.csv
Loaded ITC list (25 names) → added 'itc_countries' (Yes/No).
Unknown statuses (treated as <NA>): {'core_group_status': 1}
Saved → data\processed\data_clean.csv
Saved → outputs\eda_overview.csv

All files written under:
 - data\raw
 - data\processed
 - outputs


C:\Users\James\AppData\Local\Temp\ipykernel_28068\2688302392.py:151: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pend_mask = low.str.contains(r"\b(pending|tbc|awaiting)\b", na=False)
C:\Users\James\AppData\Local\Temp\ipykernel_28068\2688302392.py:151: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pend_mask = low.str.contains(r"\b(pending|tbc|awaiting)\b", na=False)
C:\Users\James\AppData\Local\Temp\ipykernel_28068\2688302392.py:151: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pend_mask = low.str.contains(r"\b(pending|tbc|awaiting)\b", na=False)
C:\Users\James\AppData\Local\Temp\ipykernel_28068\2688302392.py:151: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the g